# KSP Crime Intelligence — Exploratory Data Analysis
## Karnataka State Police ML Service

This notebook explores the historical crime data and census data to understand patterns, correlations, and feature distributions for ML model training.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib
import os

sns.set_theme(style='darkgrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

## 1. Load Data
Load historical cases and census data from the CSV files.

In [ ]:
cases = pd.read_csv('../data/historical_cases.csv')
census = pd.read_csv('../data/census_data.csv')
print(f'Cases: {cases.shape}, Census: {census.shape}')
cases.head()

In [ ]:
census.head()

## 2. Case Data Distribution

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
cases['DistrictID'].value_counts().sort_index().plot(kind='bar', ax=axes[0,0], title='Cases by District', color='#2C4A7C')
cases['CrimeHeadID'].value_counts().sort_index().plot(kind='bar', ax=axes[0,1], title='Cases by Crime Head', color='#C0392B')
cases['GenderID'].value_counts().plot(kind='pie', ax=axes[0,2], title='Gender Distribution', autopct='%1.1f%%', colors=['#2C4A7C', '#C9A94E'])
cases['AgeYear'].hist(bins=20, ax=axes[1,0], color='#2C4A7C', edgecolor='white')
axes[1,0].set_title('Age Distribution')
axes[1,0].set_xlabel('Age')
cases['ResponseTimeMinutes'].hist(bins=20, ax=axes[1,1], color='#C0392B', edgecolor='white')
axes[1,1].set_title('Response Time Distribution')
axes[1,1].set_xlabel('Minutes')
cases['RegisteredDate'] = pd.to_datetime(cases['RegisteredDate'])
cases['month'] = cases['RegisteredDate'].dt.month
cases['month'].value_counts().sort_index().plot(kind='bar', ax=axes[1,2], title='Cases by Month', color='#C9A94E')
plt.tight_layout()
plt.show()

## 3. District-level Crime Aggregation

In [ ]:
district_crime = cases.groupby('DistrictID').agg(
    total_cases=('CaseID', 'count'),
    repeat_pct=('IsRepeatOffender', 'mean'),
    avg_response=('ResponseTimeMinutes', 'mean'),
).reset_index()

merged = district_crime.merge(census, on='DistrictID')
merged['crime_rate_per_100k'] = merged['total_cases'] / merged['Population'] * 100000
merged.head()

In [ ]:
# Correlation matrix
corr_cols = ['total_cases', 'repeat_pct', 'avg_response', 'crime_rate_per_100k',
             'PopulationDensity', 'UrbanizationIndex', 'LiteracyRate', 'PovertyIndex', 'UnemploymentRate']
corr = merged[corr_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, fmt='.2f')
plt.title('Correlation: Crime Metrics vs Socio-Economic Indicators')
plt.tight_layout()
plt.show()

## 4. Train Risk Score Model
Using RandomForestRegressor with engineered district-level features.

In [ ]:
features = ['total_cases', 'crime_rate_per_100k', 'PopulationDensity',
            'UrbanizationIndex', 'LiteracyRate', 'avg_response', 'repeat_pct']
X = merged[features]
y = (0.15 * merged['total_cases'] / merged['total_cases'].max() +
     0.20 * merged['crime_rate_per_100k'] / merged['crime_rate_per_100k'].max() +
     0.10 * merged['PopulationDensity'] / merged['PopulationDensity'].max() +
     0.10 * merged['UrbanizationIndex'] / merged['UrbanizationIndex'].max() -
     0.15 * merged['LiteracyRate'] / merged['LiteracyRate'].max() +
     0.15 * (1 - merged['avg_response'] / merged['avg_response'].max()) +
     0.15 * merged['repeat_pct'] / merged['repeat_pct'].max())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

y_pred = rf.predict(X_test)
print(f'R² Score: {r2_score(y_test, y_pred):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}')

# Save model
os.makedirs('../trained_models', exist_ok=True)
joblib.dump(rf, '../trained_models/risk_score_model.pkl')
print('✓ risk_score_model.pkl saved')

## 5. Train Anomaly Detection Model
Using IsolationForest on individual case features.

In [ ]:
anomaly_features = ['AgeYear', 'GenderID', 'DistrictID', 'CrimeHeadID', 'ResponseTimeMinutes', 'IsRepeatOffender']
X_anom = cases[anomaly_features].fillna(0)

iso = IsolationForest(n_estimators=100, contamination=0.05, random_state=42)
iso.fit(X_anom)

preds = iso.predict(X_anom)
print(f'Anomalies detected: {(preds == -1).sum()} / {len(preds)} ({(preds == -1).mean()*100:.1f}%)')

joblib.dump(iso, '../trained_models/anomaly_model.pkl')
print('✓ anomaly_model.pkl saved')

## 6. Feature Importance Analysis

In [ ]:
importance = pd.DataFrame({'feature': features, 'importance': rf.feature_importances_})
importance = importance.sort_values('importance', ascending=False)
plt.figure(figsize=(10, 5))
sns.barplot(data=importance, x='importance', y='feature', palette='viridis')
plt.title('Risk Score Model — Feature Importance')
plt.tight_layout()
plt.show()
importance

## 7. Socio-Economic Correlation Analysis

In [ ]:
from scipy import stats
indicators = ['UnemploymentRate', 'LiteracyRate', 'PovertyIndex', 'UrbanizationIndex', 'PolicePerCapita']
cr = merged['crime_rate_per_100k'].values
for ind in indicators:
    vals = merged[ind].values
    coef, p_val = stats.pearsonr(vals, cr)
    sig = '***' if p_val < 0.01 else '**' if p_val < 0.05 else '*' if p_val < 0.1 else ''
    print(f'{ind:20s}  r = {coef:+.4f}  p = {p_val:.4f}  {sig}')

In [ ]:
print('\nEDA complete. All models trained and saved.')